In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')

project_dir = Path('.')
data_dir = project_dir / 'data_raw'
clean_dir = project_dir / 'data_clean'
clean_dir.mkdir(exist_ok=True)

print(f'Project directory: {project_dir.resolve()}')
print(f'Data directory: {data_dir.resolve()}')

In [ ]:
na_values = ['\N', '', 'NA', 'N/A', 'nan']

datasets = {
    'circuits': pd.read_csv(data_dir / 'circuits.csv', na_values=na_values),
    'constructors': pd.read_csv(data_dir / 'constructors.csv', na_values=na_values),
    'constructor_results': pd.read_csv(data_dir / 'constructor_results.csv', na_values=na_values),
    'constructor_standings': pd.read_csv(data_dir / 'constructor_standings.csv', na_values=na_values),
    'drivers': pd.read_csv(data_dir / 'drivers.csv', na_values=na_values),
    'driver_standings': pd.read_csv(data_dir / 'driver_standings.csv', na_values=na_values),
    'lap_times': pd.read_csv(data_dir / 'lap_times.csv', na_values=na_values),
    'pit_stops': pd.read_csv(data_dir / 'pit_stops.csv', na_values=na_values),
    'qualifying': pd.read_csv(data_dir / 'qualifying.csv', na_values=na_values),
    'races': pd.read_csv(data_dir / 'races.csv', na_values=na_values),
    'results': pd.read_csv(data_dir / 'results.csv', na_values=na_values),
    'seasons': pd.read_csv(data_dir / 'seasons.csv', na_values=na_values),
    'sprint_results': pd.read_csv(data_dir / 'sprint_results.csv', na_values=na_values),
    'status': pd.read_csv(data_dir / 'status.csv', na_values=na_values),
    'students': pd.read_csv(data_dir / 'students.csv', na_values=na_values),
}

for name, df in datasets.items():
    print(f'{name}: rows={len(df)}, columns={len(df.columns)}')
    display(df.head(3))
    print('-' * 80)

In [ ]:
# Clean the circuits table
circuits = datasets['circuits'].copy()
circuits['alt'] = pd.to_numeric(circuits['alt'], errors='coerce')
circuits['lat'] = pd.to_numeric(circuits['lat'], errors='coerce')
circuits['lng'] = pd.to_numeric(circuits['lng'], errors='coerce')
circuits['country'] = circuits['country'].replace({'UK': 'United Kingdom'})

circuits.head()

In [ ]:
# Clean the results table and create numeric versions of key columns
results = datasets['results'].copy()
results['milliseconds'] = pd.to_numeric(results['milliseconds'], errors='coerce')
results['points'] = pd.to_numeric(results['points'], errors='coerce')
results['grid'] = pd.to_numeric(results['grid'], errors='coerce')
results['position'] = pd.to_numeric(results['position'], errors='coerce')
results['rank'] = pd.to_numeric(results['rank'], errors='coerce')
results['statusId'] = pd.to_numeric(results['statusId'], errors='coerce')

results[['resultId', 'raceId', 'driverId', 'constructorId', 'points', 'milliseconds']].head()

In [ ]:
# Create a clean standings table and a year bracket variable
constructor_standings = datasets['constructor_standings'].copy()
constructor_standings['position'] = pd.to_numeric(constructor_standings['position'], errors='coerce')
constructor_standings['positionText'] = pd.to_numeric(constructor_standings['positionText'], errors='coerce')

races = datasets['races'].copy()
races['year'] = pd.to_numeric(races['year'], errors='coerce')
races['year_brackets'] = pd.cut(races['year'], bins=[2008, 2011, 2014, 2017, 2020, 2023, 2026], labels=['2009-2011', '2012-2014', '2015-2017', '2018-2020', '2021-2023', '2024-2026'])

constructor_standings.head()

In [ ]:
# Merge results with drivers, constructors, races, and status
drivers = datasets['drivers'].copy()
constructors = datasets['constructors'].copy()
status = datasets['status'].copy()

results_full = (results
    .merge(drivers[['driverId', 'forename', 'surname', 'nationality']], on='driverId', how='left')
    .merge(constructors[['constructorId', 'constructorRef', 'name', 'nationality']], on='constructorId', how='left')
    .merge(races[['raceId', 'year', 'round', 'name', 'circuitId', 'date']], on='raceId', how='left')
    .merge(status, on='statusId', how='left'))

results_full = results_full.rename(columns={'name_x': 'race_name', 'name_y': 'constructor_name'})
results_full[['raceId', 'year', 'round', 'race_name', 'driverId', 'forename', 'surname', 'constructorId', 'constructor_name', 'points', 'status']].head()

In [ ]:
# Driver performance summary
driver_points = (results_full
    .groupby(['forename', 'surname'], as_index=False)['points']
    .sum()
    .rename(columns={'points': 'total_points'})
    .sort_values('total_points', ascending=False)
    .head(10))

display(driver_points)

plt.figure(figsize=(10, 6))
sns.barplot(data=driver_points, x='total_points', y='surname', palette='viridis')
plt.title('Top 10 Drivers by Total Points')
plt.xlabel('Total Points')
plt.ylabel('Driver')
plt.tight_layout()

In [ ]:
# Constructor performance summary
constructor_points = (results_full
    .groupby(['constructor_name'], as_index=False)['points']
    .sum()
    .rename(columns={'points': 'total_points'})
    .sort_values('total_points', ascending=False)
    .head(10))

display(constructor_points)

plt.figure(figsize=(10, 6))
sns.barplot(data=constructor_points, x='total_points', y='constructor_name', palette='magma')
plt.title('Top 10 Constructors by Total Points')
plt.xlabel('Total Points')
plt.ylabel('Constructor')
plt.tight_layout()

In [ ]:
# Lap time and pit stop summaries
lap_times = datasets['lap_times'].copy()
pit_stops = datasets['pit_stops'].copy()

lap_times['time'] = pd.to_timedelta(lap_times['time'])
lap_times['lap_seconds'] = lap_times['time'].dt.total_seconds()

pit_stops['duration'] = pd.to_timedelta(pit_stops['duration'])
pit_stops['pit_duration_seconds'] = pit_stops['duration'].dt.total_seconds()

lap_summary = lap_times.groupby('raceId', as_index=False)['lap_seconds'].mean().rename(columns={'lap_seconds': 'avg_lap_seconds'})
pit_summary = pit_stops.groupby('raceId', as_index=False)['stop'].count().rename(columns={'stop': 'pit_stops_total'})

race_performance = lap_summary.merge(pit_summary, on='raceId', how='left')
display(race_performance.head())

In [ ]:
# Qualifying and seasonal analysis
qualifying = datasets['qualifying'].copy()
seasons = datasets['seasons'].copy()

qualifying['position'] = pd.to_numeric(qualifying['position'], errors='coerce')
qualifying['q1'] = pd.to_numeric(qualifying['q1'], errors='coerce')
qualifying['q2'] = pd.to_numeric(qualifying['q2'], errors='coerce')
qualifying['q3'] = pd.to_numeric(qualifying['q3'], errors='coerce')

season_summary = (results_full
    .groupby('year', as_index=False)['points']
    .sum()
    .rename(columns={'points': 'season_points'}))

display(season_summary.head())

plt.figure(figsize=(10, 5))
sns.lineplot(data=season_summary, x='year', y='season_points', marker='o')
plt.title('Season Points by Year')
plt.xlabel('Year')
plt.ylabel('Total Points')
plt.tight_layout()

In [ ]:
# Example recoding from the student dataset
students = datasets['students'].copy()
students['lettergrade'] = pd.cut(
    students['numericgrade'],
    bins=[0, 59, 69, 79, 89, 100],
    labels=['F', 'D', 'C', 'B', 'A'],
    include_lowest=True
)

students.head()

In [ ]:
# Save cleaned outputs
circuits.to_csv(clean_dir / 'circuits.csv', index=False)
results.to_csv(clean_dir / 'results.csv', index=False)
constructor_standings.to_csv(clean_dir / 'constructor_standings.csv', index=False)
races.to_csv(clean_dir / 'races.csv', index=False)
students.to_csv(clean_dir / 'students.csv', index=False)

print('Cleaned files saved to:', clean_dir.resolve())

## Interpretation

This project combines the core lecture material into a coherent workflow:
- row-level inspection and dtype checks
- recoding and cleaning values like `alt`, `milliseconds`, `country`, and `position`
- merging multiple tables to create richer analysis data
- grouping and descriptive statistics by driver, constructor, and season
- plotting results for a more polished report

You can extend this notebook by adding more custom questions, such as: whether championship points increase with qualifying position, which races have the highest average pit-stop duration, or how constructor standings change over time.